# Gink's mind — an open-weights chat model on a borrowed GPU

Qwen2.5-7B-Instruct (Apache-2.0) in 4-bit, served to the oracle through an
OpenAI-compatible endpoint, **with tool calling** — so Gink can search the
web, read pages, draw and film from this model exactly as he would from a
paid one.

Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**.

The 7B model answers in roughly ten to twenty seconds on a T4. Pick the 14B
in the form if you have an L4 or an A100.


In [ ]:
#@title EGregoRA · Gink's mind — an open-weights chat model { display-mode: "form" }
#@markdown Runs **Qwen2.5-7B-Instruct** (Apache-2.0, open weights) on this
#@markdown machine's GPU and lends it to the oracle. Runtime → T4 GPU → Run all.
#@markdown Leave the tab open. Nothing about your Google account is shared.

SITE   = "https://egregora.hatchable.site"   #@param {type:"string"}
MODEL  = "Qwen/Qwen2.5-7B-Instruct"          #@param ["Qwen/Qwen2.5-7B-Instruct", "meta-llama/Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "Qwen/Qwen2.5-14B-Instruct"]
LABEL  = "Gink's mind"                       #@param {type:"string"}
SECRET = ""                                  #@param {type:"string"}
FOUR_BIT = True                              #@param {type:"boolean"}

import subprocess, sys, os, threading, time, json, uuid, re, shutil, urllib.request
assert shutil.which("nvidia-smi"), "No GPU. Runtime - Change runtime type - T4 GPU, then run again."
GPU = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(GPU)

subprocess.run([sys.executable,"-m","pip","install","-q",
                "transformers>=4.45","accelerate","bitsandbytes","sentencepiece","flask"], check=True)

In [ ]:
#@title Load the model and serve it
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from flask import Flask, request, jsonify

print("loading", MODEL, "— a few minutes the first time")
tok = AutoTokenizer.from_pretrained(MODEL)
kw = dict(device_map="auto", torch_dtype=torch.float16)
if FOUR_BIT:
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, **kw)
model.eval()
print("loaded.")

LOCK = threading.Lock()

def to_openai_tool_calls(text):
    """Qwen emits <tool_call>{...}</tool_call>; turn that into OpenAI's shape."""
    calls = []
    for m in re.finditer(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, re.S):
        try: obj = json.loads(m.group(1))
        except Exception: continue
        calls.append({"id": "call_" + uuid.uuid4().hex[:8], "type": "function",
                      "function": {"name": obj.get("name",""),
                                   "arguments": json.dumps(obj.get("arguments", {}))}})
    cleaned = re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.S).strip()
    return cleaned, calls

def generate(messages, tools, temperature, max_tokens):
    chat = []
    for m in messages:
        role = m.get("role", "user")
        if role == "tool":
            chat.append({"role": "tool", "name": m.get("name",""),
                         "content": str(m.get("content",""))[:12000]})
        else:
            entry = {"role": role, "content": str(m.get("content") or "")}
            if m.get("tool_calls"):
                entry["tool_calls"] = m["tool_calls"]
            chat.append(entry)
    try:
        ids = tok.apply_chat_template(chat, tools=tools or None,
                                      add_generation_prompt=True, return_tensors="pt")
    except Exception:
        ids = tok.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt")
    ids = ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=int(max_tokens),
                             do_sample=temperature > 0.05, temperature=max(0.05, temperature),
                             top_p=0.9, repetition_penalty=1.05,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()

app = Flask(__name__)

@app.get("/health")
def health(): return jsonify(ok=True, gpu=GPU, model=MODEL, caps="chat")

@app.post("/v1/chat/completions")
def chat():
    if SECRET and request.headers.get("x-egregora-secret") != SECRET:
        return jsonify(error="forbidden"), 403
    b = request.get_json(force=True, silent=True) or {}
    msgs = b.get("messages") or []
    if not msgs: return jsonify(error="messages required"), 400
    started = time.time()
    with LOCK:
        text = generate(msgs, b.get("tools"), float(b.get("temperature", 0.7)),
                        int(b.get("max_tokens", 1000)))
    content, calls = to_openai_tool_calls(text)
    msg = {"role": "assistant", "content": content}
    if calls: msg["tool_calls"] = calls
    print(f"{time.time()-started:.1f}s  {len(content)} chars  {len(calls)} tool calls", flush=True)
    return jsonify(id="chatcmpl-" + uuid.uuid4().hex[:12], object="chat.completion",
                   model=MODEL, choices=[{"index": 0, "message": msg,
                                          "finish_reason": "tool_calls" if calls else "stop"}])

threading.Thread(target=lambda: app.run(host="0.0.0.0", port=8712, threaded=True),
                 daemon=True).start()
time.sleep(2)
print("mind up on :8712")

In [ ]:
#@title Open the tunnel and lend the mind to the order
if not os.path.exists("/content/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/content/cloudflared")
    os.chmod("/content/cloudflared", 0o755)

log = open("/content/tunnel.log", "w")
subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate",
                  "--url", "http://127.0.0.1:8712"], stdout=log, stderr=subprocess.STDOUT)

PUBLIC = None
for _ in range(60):
    time.sleep(2)
    try: txt = open("/content/tunnel.log").read()
    except FileNotFoundError: txt = ""
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", txt)
    if m: PUBLIC = m.group(0); break
assert PUBLIC, "the tunnel did not come up — run this cell again"
print("public endpoint:", PUBLIC)

def post(payload):
    req = urllib.request.Request(SITE + "/api/colab",
        data=json.dumps(payload).encode(),
        headers={"content-type": "application/json", "x-egregora-secret": SECRET})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode())

print(post({"action": "register", "label": LABEL, "endpoint": PUBLIC, "gpu": GPU,
            "caps": "chat", "model": MODEL, "secret": SECRET}))
print("registered — the oracle now thinks with", MODEL)

try:
    while True:
        time.sleep(120)
        try: post({"action": "heartbeat", "endpoint": PUBLIC, "secret": SECRET})
        except Exception as e: print("heartbeat failed:", e)
except KeyboardInterrupt:
    post({"action": "retire", "endpoint": PUBLIC, "secret": SECRET})
    print("retired.")